# BERTopic Analysis with OpenAI Representation

This notebook performs topic modeling on climate-related transcripts using BERTopic with multiple representation models including OpenAI's GPT-4o-mini for generating human-readable topic labels.

## Overview

The pipeline consists of the following steps:
1. Load and prepare documents
2. Generate or load document embeddings
3. Reduce embeddings for visualization
4. Configure clustering and vectorization
5. Set up topic representation models
6. Fit BERTopic model and extract topics
7. Visualize and save results

## 1. Import Required Libraries

We import all necessary libraries for:
- **Data manipulation**: pandas
- **Topic modeling**: BERTopic and its components
- **Embeddings**: SentenceTransformer for multilingual embeddings
- **Dimensionality reduction**: UMAP
- **Clustering**: HDBSCAN
- **Text vectorization**: CountVectorizer
- **Topic representation**: KeyBERT, MMR, and OpenAI
- **Utilities**: NLTK for stopwords, pickle for caching, dotenv for API keys

In [1]:
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
import hdbscan
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, OpenAI
from nltk.corpus import stopwords
import nltk
from pathlib import Path
import pickle
import os
from dotenv import load_dotenv
from openai import OpenAI as OpenAIClient

# Load environment variables from .env file
load_dotenv()

C:\Users\sile9\anaconda3\envs\bertopic_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## 2. Configure Project Paths

Set up the project root directory path. This ensures all file paths work correctly regardless of where the notebook is located.

**Options:**
- If your notebook is in the project root, use `Path.cwd()`
- If your notebook is in a subdirectory (e.g., `notebooks/`), use `Path.cwd().parent`
- Or manually specify the absolute path to your project folder

In [2]:
# Configure project root path
# Option 1: Notebook is in project root
# project_root = Path.cwd()

# Option 2: Notebook is in a subdirectory (e.g., notebooks/)
project_root = Path.cwd().parent

# Option 3: Manually specify absolute path
# project_root = Path('/path/to/your/project')

print(f"Project root: {project_root}")

# Define data and output directories
data_dir = project_root / 'Data'
output_dir = project_root / 'Outputs'

# Create directories if they don't exist
data_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {data_dir}")
print(f"Output directory: {output_dir}")

Project root: c:\Users\sile9\Documents\projects\parldebates_analysis
Data directory: c:\Users\sile9\Documents\projects\parldebates_analysis\Data
Output directory: c:\Users\sile9\Documents\projects\parldebates_analysis\Outputs


## 3. Load Documents

Read the preprocessed CSV file containing climate-related transcripts. Each row represents a document that will be analyzed for topic modeling.

In [3]:
df = pd.read_csv(data_dir / 'transcripts_climate_for_topic_modeling.csv')
text_column = 'transcript_text'
documents = df[text_column].tolist()

print(f"Loaded {len(documents)} documents")
print(f"First document preview: {documents[0][:200]}...")

Loaded 6396 documents
First document preview: Die im Postulat gestellte Frage ist natürlich eine wichtige Frage. Der Bundesrat empfiehlt Ihnen deshalb die Annahme des Postulates. Sie wissen, dass der Bundesrat zu dieser Frage zwischenzeitlich Stu...


## 4. Generate or Load Document Embeddings

Embeddings are dense vector representations of documents that capture their semantic meaning. We use the **paraphrase-multilingual-mpnet-base-v2** model which works well for German and French texts.

To save computation time, we cache embeddings to disk. If they exist, we load them; otherwise, we compute and save them.

In [4]:
print("Calculating or loading embeddings...")
embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

emb_path = data_dir / 'embeddings.pickle'
if emb_path.exists():
    print("Loading existing embeddings from disk...")
    with emb_path.open('rb') as handle:
        embeddings = pickle.load(handle)
else:
    print("Computing embeddings (this may take a few minutes)...")
    embeddings = embedding_model.encode(documents, show_progress_bar=True)
    emb_path.parent.mkdir(parents=True, exist_ok=True)
    with emb_path.open('wb') as handle:
        pickle.dump(embeddings, handle, protocol=pickle.HIGHEST_PROTOCOL)
    print("Embeddings saved to disk.")

print(f"Embeddings shape: {embeddings.shape}")

Calculating or loading embeddings...
Loading existing embeddings from disk...
Embeddings shape: (6396, 768)


<positron-console-cell-5>:8: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.


## 5. Pre-reduce Embeddings for Visualization (2D)

We use **UMAP** (Uniform Manifold Approximation and Projection) to reduce the high-dimensional embeddings to 2D for visualization purposes. This allows us to plot documents in a 2D space where similar documents are close together.

**Parameters:**
- `n_neighbors=15`: Controls local vs. global structure (higher = more global)
- `n_components=2`: Target dimensionality (2D for visualization)
- `min_dist=0.0`: Minimum distance between points in low-dimensional space
- `metric='cosine'`: Distance metric suitable for text embeddings
- `random_state=42`: For reproducibility

In [5]:
print("Reducing embeddings for visualization...")
reduced_embeddings = UMAP(
    n_neighbors=15, 
    n_components=2, 
    min_dist=0.0, 
    metric='cosine', 
    random_state=42
).fit_transform(embeddings)

print(f"Reduced embeddings shape: {reduced_embeddings.shape}")

Reducing embeddings for visualization...
Reduced embeddings shape: (6396, 2)


## 6. Configure UMAP Model for Topic Modeling (5D)

For the actual topic modeling, we reduce to 5 dimensions instead of 2. This preserves more information while still reducing dimensionality enough for effective clustering.

5 dimensions strikes a balance between:
- Computational efficiency
- Preserving semantic relationships
- Effective clustering by HDBSCAN

In [6]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

## 7. Configure HDBSCAN Clustering Model

**HDBSCAN** (Hierarchical Density-Based Spatial Clustering of Applications with Noise) is used to identify clusters in the reduced embedding space. Each cluster represents a topic.

**Parameters:**
- `min_cluster_size=50`: Minimum documents per topic (~1.7% of dataset). Smaller values create more granular topics.
- `min_samples=5`: How conservative the clustering is. Lower values allow more points to be clustered.
- `metric='euclidean'`: Distance metric for clustering
- `cluster_selection_method='eom'`: Excess of Mass method for selecting clusters
- `prediction_data=True`: Allows predicting topics for new documents

In [7]:
hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=50,
    min_samples=5,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

## 8. Configure CountVectorizer with Stopwords

**CountVectorizer** converts documents into a bag-of-words representation for topic description. We remove common stopwords in German, French, and Swiss German to focus on meaningful words.

**Parameters:**
- `min_df=2`: Ignore words appearing in fewer than 2 documents
- `max_df=0.85`: Ignore words appearing in more than 85% of documents (too common)
- `ngram_range=(1, 2)`: Consider both single words and two-word phrases
- `stop_words`: Custom list of German, French, and Swiss German stopwords

In [8]:
# Download stopwords (only needed once)
nltk.download('stopwords', quiet=True)

# Get German and French stopwords
german_stopwords = set(stopwords.words('german'))
french_stopwords = set(stopwords.words('french'))
all_stopwords = german_stopwords.union(french_stopwords)

# Add Swiss German specific stopwords
swiss_german_stopwords = {'dass'}  # Add more Swiss German words here if needed
all_stopwords = all_stopwords.union(swiss_german_stopwords)

vectorizer_model = CountVectorizer(
    min_df=2,
    max_df=0.85,
    ngram_range=(1, 2),
    stop_words=list(all_stopwords)
)

print(f"Total stopwords: {len(all_stopwords)}")

Total stopwords: 386


## 9. Configure Topic Representation Models

BERTopic allows multiple representation models to generate different perspectives on topic labels:

1. **KeyBERT**: Extracts keywords most similar to the topic using cosine similarity
2. **MMR (Maximal Marginal Relevance)**: Balances keyword relevance and diversity (`diversity=0.3`)
3. **OpenAI GPT-4o-mini**: Generates human-readable topic labels in English using representative documents and keywords

The OpenAI model receives:
- Top keywords from the topic
- 5 representative documents (truncated to 200 words each)
- A prompt to generate a concise English label (max 5 words)

In [9]:
# KeyBERT and MMR representation models
keybert = KeyBERTInspired()
mmr = MaximalMarginalRelevance(diversity=0.3)

# OpenAI representation model
print("Setting up OpenAI representation model...")
openai_api_key = os.getenv("OPENAI_API_KEY")

# Create OpenAI client
client = OpenAIClient(api_key=openai_api_key)

# Prompt for OpenAI to generate topic labels
openai_prompt = """
I have a topic described by the following keywords: [KEYWORDS]

The topic contains these representative documents:
[DOCUMENTS]

Based on the keywords and documents above, generate a short, descriptive label for this topic in English (maximum 5 words). Only return the label itself in English, nothing else.
"""

openai_model = OpenAI(
    client=client,
    model="gpt-4o-mini",  # Cost-effective model
    prompt=openai_prompt,
    chat=True,
    nr_docs=5,  # Number of representative documents to include
    doc_length=200,  # Maximum words per document
    tokenizer="whitespace"  # Required: method to tokenize documents
)

representation_model = {
    "KeyBERT": keybert,
    "MMR": mmr,
    "OpenAI": openai_model,
}

print("Representation models configured.")

Setting up OpenAI representation model...
Representation models configured.


## 10. Initialize and Fit BERTopic Model

Now we bring all components together and fit the BERTopic model:

**Process:**
1. Use pre-computed embeddings to represent documents
2. Apply UMAP to reduce to 5D
3. Apply HDBSCAN to cluster similar documents
4. Use CountVectorizer to extract keywords from each cluster
5. Apply representation models to generate topic labels

**Parameters:**
- `nr_topics="auto"`: Automatically determine the number of topics (no merging)
- `top_n_words=10`: Show top 10 words per topic
- `verbose=True`: Display progress information

In [10]:
print("Fitting BERTopic model...")
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model,
    nr_topics="auto",
    top_n_words=10,
    verbose=True
)

topics, probs = topic_model.fit_transform(documents, embeddings)

print("\nModel fitting complete!")

2026-01-28 12:52:33,883 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Fitting BERTopic model...


2026-01-28 12:52:40,229 - BERTopic - Dimensionality - Completed ✓
2026-01-28 12:52:40,230 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-01-28 12:52:40,337 - BERTopic - Cluster - Completed ✓
2026-01-28 12:52:40,337 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-01-28 12:52:42,972 - BERTopic - Representation - Completed ✓
2026-01-28 12:52:42,974 - BERTopic - Topic reduction - Reducing number of topics
2026-01-28 12:52:42,981 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 16/16 [00:18<00:00,  1.15s/it]
2026-01-28 12:53:15,860 - BERTopic - Representation - Completed ✓
2026-01-28 12:53:15,876 - BERTopic - Topic reduction - Reduced number of topics from 26 to 16



Model fitting complete!


## 11. Add Topics to DataFrame

We add the assigned topic and probability to each document in the original dataframe. Topic `-1` indicates outliers that didn't fit well into any cluster.

In [11]:
df['topic'] = topics
df['topic_probability'] = probs

print(df[['transcript_text', 'topic', 'topic_probability']].head())

                                     transcript_text  topic  topic_probability
0  Die im Postulat gestellte Frage ist natürlich ...     14                1.0
1  Wir alle wissen, dass der Bundesrat seit der V...     14                1.0
2  Das Postulat geht einerseits zu weit, und ande...     -1                0.0
3  Mit der Teilrevision des Mineralölsteuergesetz...      7                1.0
4  Die Lösung, die Sie jetzt vorschlagen, indem S...     -1                0.0


## 12. Display Topic Analysis Results

Let's examine the topics discovered by the model, including:
- Total number of topics (excluding outliers)
- Distribution of documents across topics
- Detailed topic information with all representation models

In [12]:
print(f"\nNumber of topics found: {len(set(topics)) - 1}")  # -1 for outlier topic
print("\nTopic distribution:")
print(df['topic'].value_counts())

# Get topic information with multiple representations
topic_info = topic_model.get_topic_info()
print("\nTopic Information:")
print(topic_info)


Number of topics found: 15

Topic distribution:
topic
 0     2232
-1     1902
 1      678
 2      303
 3      288
 4      241
 5      116
 6      115
 7       85
 8       75
 9       71
 10      69
 11      66
 12      52
 13      52
 14      51
Name: count, dtype: int64

Topic Information:
    Topic  Count                                               Name  \
0      -1   1902  -1_anlagen_wasserkraft_erneuerbaren energien_g...   
1       0   2232  0_wasserkraft_anlagen_energiestrategie_terawat...   
2       1    678           1_klima_gletscher_klimawandel_emissionen   
3       2    303  2_biodiversität_biodiversité_landwirtschaft_fl...   
4       3    288         3_flugticketabgabe_luftfahrt_taxe_aviation   
5       4    241  4_verkehr_verkehrs_öffentlichen verkehrs_véhic...   
6       5    116               5_kernkraftwerke_ensi_akw_nucléaires   
7       6    115          6_axpo_anlagen_eigenhandel_verwaltungsrat   
8       7     85       7_mineralölsteuer_rückerstattung_verkehr_naf 

## 13. Save Results to CSV

Export the results for further analysis:
- **documents_with_topics.csv**: Original data with assigned topics
- **topic_info.csv**: Topic labels and keywords from all representation models

In [13]:
df.to_csv(data_dir / '12_documents_with_topics.csv', index=False)
topic_info.to_csv(data_dir / '12_topic_info.csv', index=False)

## 14. Create Interactive Visualizations

BERTopic provides several interactive visualizations:

1. **Document Visualization**: 2D plot showing all documents colored by topic
2. **Intertopic Distance Map**: Shows relationships between topics in 2D space
3. **Topic Hierarchy**: Dendrogram showing how topics relate hierarchically
4. **Topic Barchart**: Top words for each topic (showing top 10 topics)

All visualizations are saved as interactive HTML files.

In [14]:
print("\nCreating visualizations...")

# Visualize documents with pre-reduced embeddings
fig_docs = topic_model.visualize_documents(
    documents, 
    reduced_embeddings=reduced_embeddings,
    hide_document_hover=False,
    hide_annotations=False
)
fig_docs.write_html(output_dir / "12_topic_documents_visualization.html")
print("  ✓ Document visualization saved")

# Visualize intertopic distance map
fig_topics = topic_model.visualize_topics()
fig_topics.write_html(output_dir / "12_intertopic_distance_map.html")
print("  ✓ Intertopic distance map saved")

# Visualize topic hierarchy
fig_hierarchy = topic_model.visualize_hierarchy()
fig_hierarchy.write_html(output_dir / "12_topic_hierarchy.html")
print("  ✓ Topic hierarchy saved")

# Visualize barchart of top words per topic
fig_barchart = topic_model.visualize_barchart(top_n_topics=10)
fig_barchart.write_html(output_dir / "12_topic_barchart.html")
print("  ✓ Topic barchart saved")

print(f"\nProcessing complete! Check the {output_dir} folder for visualizations.")


Creating visualizations...
  ✓ Document visualization saved
  ✓ Intertopic distance map saved
  ✓ Topic hierarchy saved
  ✓ Topic barchart saved

Processing complete! Check the c:\Users\sile9\Documents\projects\parldebates_analysis\Outputs folder for visualizations.


## Summary

This notebook performed topic modeling on climate transcripts using:
- Multilingual sentence embeddings
- UMAP for dimensionality reduction
- HDBSCAN for density-based clustering
- Multiple representation models (KeyBERT, MMR, OpenAI)

The OpenAI integration provides human-readable English labels that summarize each topic based on keywords and representative documents.

**Next steps:**
- Examine the visualizations in the Outputs folder
- Review topic_info.csv to see all topic representations
- Analyze documents_with_topics.csv to understand topic assignments
- Consider adjusting `min_cluster_size` or other parameters if topics are too broad or too granular